# EHR-M-GAN: 3-Model Training Efficiency & Quality Analysis on Google Colab

This notebook allows you to load checkpoints from **all 3 versions** of EHR-M-GAN and compare their performance, convergence rate, and continuous/discrete generation quality side-by-side:

1. **Model 1: Standard M3GAN (Baseline)**: The basic Coupled VAE-GAN implementation.
2. **Model 2: Neo M3GAN V1**: Introduced Temporal Self-Attention and synchronized generators.
3. **Model 3: Neo M3GAN V2 (Ours)**: Added **3-layer MLP Mapping Networks** in G, **Minibatch Standard Deviation** in D, **KL weight annealing** in Phase 1, and **numerical autograd stability** to fully prevent mode collapse.

---

In [ ]:
# 1. Mount Google Drive to load datasets/checkpoints
from google.colab import drive
import os
import numpy as np
import pickle
import pandas as pd
import torch

# drive.mount('/content/drive')

# 2. Create local directory structure
os.makedirs('Data/mimic', exist_ok=True)
os.makedirs('Output/analysis', exist_ok=True)

### ⚠️ Data Upload Instructions:
Please upload your patient data (`vital_sign_24hrs.pkl` and `med_interv_24hrs.pkl`) to your Google Drive or upload them directly to the Colab files section under `Data/mimic/`.

Once uploaded, the cell below will verify and load the patient data.

In [ ]:
# Load and normalize real dataset for validation comparison
vital_path = 'Data/mimic/vital_sign_24hrs.pkl'
med_path = 'Data/mimic/med_interv_24hrs.pkl'

if not os.path.exists(vital_path) or not os.path.exists(med_path):
    print("❌ ERROR: Please upload 'vital_sign_24hrs.pkl' and 'med_interv_24hrs.pkl' to 'Data/mimic/' folder first!")
else:
    with open(vital_path, 'rb') as f:
        real_c = pickle.load(f)
    with open(med_path, 'rb') as f:
        real_d = pickle.load(f)
        
    # Sanitize and normalize
    real_d = np.nan_to_num(np.clip(real_d, 0.0, 1.0), nan=0.0)
    real_c = np.nan_to_num(real_c, nan=0.0)
    
    min_val = np.min(real_c, axis=(0, 1))
    max_val = np.max(real_c, axis=(0, 1))
    range_val = max_val - min_val
    range_val[range_val == 0.0] = 1e-6
    real_c_norm = (real_c - min_val) / range_val
    
    print(f"✅ Successfully loaded dataset!")
    print(f"Continuous shape: {real_c_norm.shape} | Discrete shape: {real_d.shape}")

In [ ]:
%%writefile networks_standard.py
import torch
import torch.nn as nn


class VAE_Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers=3):
        super(VAE_Encoder, self).__init__()
        # Avoid dropout error when num_layers = 1
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x_t, hidden_state):
        # x_t shape: [batch_size, 1, input_dim]
        out, new_hidden = self.lstm(x_t, hidden_state)
        out_squeeze = out.squeeze(1)

        mu = self.fc_mu(out_squeeze)
        logvar = self.fc_logvar(out_squeeze)

        # Defensive clamps to prevent numerical instability
        mu = torch.clamp(mu, min=-20.0, max=20.0)
        logvar = torch.clamp(logvar, min=-20.0, max=20.0)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z, mu, logvar, new_hidden


class VAE_Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim, num_layers=3):
        super(VAE_Decoder, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(latent_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z_t, hidden_state):
        # z_t shape: [batch_size, 1, latent_dim]
        out, new_hidden = self.lstm(z_t, hidden_state)
        logits = self.fc_out(out.squeeze(1))
        reconstruction = torch.sigmoid(logits)
        return reconstruction, logits, new_hidden


class AutoregressiveVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, enc_layers, dec_layers, time_steps):
        super(AutoregressiveVAE, self).__init__()
        self.time_steps = time_steps
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        # cVAE Architecture: Input Encoder = Original Input (x_t) + Error (x_hat) -> dim = input_dim * 2
        self.encoder = VAE_Encoder(input_dim * 2, hidden_dim, latent_dim, enc_layers)
        self.decoder = VAE_Decoder(latent_dim, hidden_dim, input_dim, dec_layers)

    def forward(self, x):
        batch_size = x.size(0)
        device = x.device

        # Initialize hidden states
        enc_hidden = (torch.zeros(self.encoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.encoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))
        dec_hidden = (torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))

        c_prev = torch.zeros(batch_size, self.input_dim, device=device)

        rec_list, logits_list, mu_list, logvar_list, z_list = [], [], [], [], []

        for t in range(self.time_steps):
            x_t = x[:, t, :]
            c_sigmoid = torch.sigmoid(c_prev)

            # Calculate residual error following VRNN logic
            x_hat = x_t - c_sigmoid

            # Pass through encoder for each timestep
            enc_in = torch.cat([x_t, x_hat], dim=1).unsqueeze(1)
            z_t, mu_t, logvar_t, enc_hidden = self.encoder(enc_in, enc_hidden)

            # Decode to get output for the next iteration
            z_in = z_t.unsqueeze(1)
            rec_t, logits_t, dec_hidden = self.decoder(z_in, dec_hidden)

            c_prev = logits_t  # Save logits (pre-sigmoid output)

            rec_list.append(rec_t.unsqueeze(1))
            logits_list.append(logits_t.unsqueeze(1))
            mu_list.append(mu_t.unsqueeze(1))
            logvar_list.append(logvar_t.unsqueeze(1))
            z_list.append(z_t.unsqueeze(1))

        return torch.cat(rec_list, dim=1), torch.cat(logits_list, dim=1), \
            torch.cat(mu_list, dim=1), torch.cat(logvar_list, dim=1), torch.cat(z_list, dim=1)

    def reconstruct_decoder(self, z_seq):
        """Function used for GAN Generator when sequence noise z is already provided"""
        batch_size = z_seq.size(0)
        device = z_seq.device
        dec_hidden = (torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))

        rec_list, logits_list = [], []
        for t in range(self.time_steps):
            z_t = z_seq[:, t, :].unsqueeze(1)
            rec_t, logits_t, dec_hidden = self.decoder(z_t, dec_hidden)
            rec_list.append(rec_t.unsqueeze(1))
            logits_list.append(logits_t.unsqueeze(1))

        return torch.cat(rec_list, dim=1), torch.cat(logits_list, dim=1)


class SequenceDiscriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim, time_steps, num_layers=3):
        super(SequenceDiscriminator, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        # Equivalent to tf.layers.flatten(outputs)
        self.fc = nn.Linear(hidden_dim * time_steps, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out_flat = torch.flatten(out, start_dim=1)
        logits = self.fc(out_flat).squeeze(-1)  # Output a single score
        return logits, out


class BilateralLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(BilateralLSTMCell, self).__init__()
        self.hidden_dim = hidden_dim
        self.linear = nn.Linear(input_dim + hidden_dim + hidden_dim, 4 * hidden_dim, bias=False)

    def forward(self, x, h_self, c_self, h_coupled):
        combined = torch.cat([x, h_self, h_coupled], dim=1)
        gates = self.linear(combined)
        i_gate, f_gate, o_gate, c_tilde = gates.chunk(4, dim=1)

        i = torch.sigmoid(i_gate)
        f = torch.sigmoid(f_gate)
        o = torch.sigmoid(o_gate)
        c_ = torch.tanh(c_tilde)

        c_next = f * c_self + i * c_
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class BilateralGenerator(nn.Module):
    def __init__(self, noise_dim, hidden_dim, latent_dim, num_layers=3):
        super(BilateralGenerator, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        self.cells = nn.ModuleList([
            BilateralLSTMCell(
                input_dim=noise_dim if i == 0 else hidden_dim,
                hidden_dim=hidden_dim
            ) for i in range(num_layers)
        ])
        self.fc_out = nn.Linear(hidden_dim, latent_dim)

    def forward(self, noise_seq, h_coupled_states):
        batch_size, time_steps, _ = noise_seq.size()
        h_states = [torch.zeros(batch_size, self.hidden_dim, device=noise_seq.device) for _ in range(self.num_layers)]
        c_states = [torch.zeros(batch_size, self.hidden_dim, device=noise_seq.device) for _ in range(self.num_layers)]
        outputs = []

        for t in range(time_steps):
            x_t = noise_seq[:, t, :]
            for i in range(self.num_layers):
                h_cpl = h_coupled_states[i]
                h_states[i], c_states[i] = self.cells[i](x_t, h_states[i], c_states[i], h_cpl)
                x_t = h_states[i]

            out_t = torch.sigmoid(self.fc_out(x_t))
            outputs.append(out_t.unsqueeze(1))

        return torch.cat(outputs, dim=1), h_states


class JointGenerator(nn.Module):
    def __init__(self, c_noise_dim, d_noise_dim, hidden_dim, c_latent_dim, d_latent_dim, num_layers=3):
        super(JointGenerator, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        
        self.c_gen = BilateralGenerator(c_noise_dim, hidden_dim, c_latent_dim, num_layers)
        self.d_gen = BilateralGenerator(d_noise_dim, hidden_dim, d_latent_dim, num_layers)

    def forward(self, noise_c, noise_d):
        """
        Implements the step-by-step bilateral coupling required by EHR-M-GAN.
        """
        batch_size, time_steps, _ = noise_c.size()
        device = noise_c.device

        # Initialize hidden and cell states for both streams
        c_h = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        c_c = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        d_h = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        d_c = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]

        fake_z_c_list = []
        fake_z_d_list = []

        for t in range(time_steps):
            # 1. Capture current noise inputs
            noise_c_t = noise_c[:, t, :]
            noise_d_t = noise_d[:, t, :]

            # 2. Coupled inputs come from the OTHER stream's PREVIOUS hidden state
            # (In Step 1, these are the initial zeros)
            c_h_coupled = d_h
            d_h_coupled = c_h

            # 3. Step C Generator Layers
            c_x = noise_c_t
            for i in range(self.num_layers):
                c_h[i], c_c[i] = self.c_gen.cells[i](c_x, c_h[i], c_c[i], c_h_coupled[i])
                c_x = c_h[i]
            
            z_c_t = torch.sigmoid(self.c_gen.fc_out(c_x))
            fake_z_c_list.append(z_c_t.unsqueeze(1))

            # 4. Step D Generator Layers
            d_x = noise_d_t
            for i in range(self.num_layers):
                d_h[i], d_c[i] = self.d_gen.cells[i](d_x, d_h[i], d_c[i], d_h_coupled[i])
                d_x = d_h[i]
            
            z_d_t = torch.sigmoid(self.d_gen.fc_out(d_x))
            fake_z_d_list.append(z_d_t.unsqueeze(1))

        fake_z_c = torch.cat(fake_z_c_list, dim=1)
        fake_z_d = torch.cat(fake_z_d_list, dim=1)

        return fake_z_c, fake_z_d

In [ ]:
%%writefile networks_v1.py
import torch
import torch.nn as nn
from torch.nn.utils import spectral_norm

class TemporalSelfAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalSelfAttention, self).__init__()
        self.query = nn.Linear(hidden_dim, hidden_dim // 8)
        self.key = nn.Linear(hidden_dim, hidden_dim // 8)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x shape: (batch_size, time_steps, hidden_dim)
        batch_size, time_steps, hidden_dim = x.size()
        
        proj_query = self.query(x) # B x T x C (where C = hidden_dim // 8)
        proj_key = self.key(x).permute(0, 2, 1) # B x C x T
        
        energy = torch.bmm(proj_query, proj_key) # B x T x T
        attention = torch.softmax(energy, dim=-1)
        
        proj_value = self.value(x) # B x T x H
        out = torch.bmm(attention, proj_value) # B x T x H
        
        out = self.gamma * out + x
        return out


class VAE_Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers=3):
        super(VAE_Encoder, self).__init__()
        # Avoid dropout error when num_layers = 1
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x_t, hidden_state):
        # x_t shape: [batch_size, 1, input_dim]
        out, new_hidden = self.lstm(x_t, hidden_state)
        out_squeeze = out.squeeze(1)

        mu = self.fc_mu(out_squeeze)
        logvar = self.fc_logvar(out_squeeze)

        # Defensive clamps to prevent numerical instability
        mu = torch.clamp(mu, min=-20.0, max=20.0)
        logvar = torch.clamp(logvar, min=-20.0, max=20.0)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z, mu, logvar, new_hidden


class VAE_Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim, num_layers=3):
        super(VAE_Decoder, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(latent_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z_t, hidden_state):
        # z_t shape: [batch_size, 1, latent_dim]
        out, new_hidden = self.lstm(z_t, hidden_state)
        logits = self.fc_out(out.squeeze(1))
        reconstruction = torch.sigmoid(logits)
        return reconstruction, logits, new_hidden


class AutoregressiveVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, enc_layers, dec_layers, time_steps):
        super(AutoregressiveVAE, self).__init__()
        self.time_steps = time_steps
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        # cVAE Architecture: Input Encoder = Original Input (x_t) + Error (x_hat) -> dim = input_dim * 2
        self.encoder = VAE_Encoder(input_dim * 2, hidden_dim, latent_dim, enc_layers)
        self.decoder = VAE_Decoder(latent_dim, hidden_dim, input_dim, dec_layers)

    def forward(self, x):
        batch_size = x.size(0)
        device = x.device

        # Initialize hidden states
        enc_hidden = (torch.zeros(self.encoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.encoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))
        dec_hidden = (torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))

        c_prev = torch.zeros(batch_size, self.input_dim, device=device)

        rec_list, logits_list, mu_list, logvar_list, z_list = [], [], [], [], []

        for t in range(self.time_steps):
            x_t = x[:, t, :]
            c_sigmoid = torch.sigmoid(c_prev)

            # Calculate residual error following VRNN logic
            x_hat = x_t - c_sigmoid

            # Pass through encoder for each timestep
            enc_in = torch.cat([x_t, x_hat], dim=1).unsqueeze(1)
            z_t, mu_t, logvar_t, enc_hidden = self.encoder(enc_in, enc_hidden)

            # Decode to get output for the next iteration
            z_in = z_t.unsqueeze(1)
            rec_t, logits_t, dec_hidden = self.decoder(z_in, dec_hidden)

            c_prev = logits_t  # Save logits (pre-sigmoid output)

            rec_list.append(rec_t.unsqueeze(1))
            logits_list.append(logits_t.unsqueeze(1))
            mu_list.append(mu_t.unsqueeze(1))
            logvar_list.append(logvar_t.unsqueeze(1))
            z_list.append(z_t.unsqueeze(1))

        return torch.cat(rec_list, dim=1), torch.cat(logits_list, dim=1), \
            torch.cat(mu_list, dim=1), torch.cat(logvar_list, dim=1), torch.cat(z_list, dim=1)

    def reconstruct_decoder(self, z_seq):
        """Function used for GAN Generator when sequence noise z is already provided"""
        batch_size = z_seq.size(0)
        device = z_seq.device
        dec_hidden = (torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))

        rec_list, logits_list = [], []
        for t in range(self.time_steps):
            z_t = z_seq[:, t, :].unsqueeze(1)
            rec_t, logits_t, dec_hidden = self.decoder(z_t, dec_hidden)
            rec_list.append(rec_t.unsqueeze(1))
            logits_list.append(logits_t.unsqueeze(1))

        return torch.cat(rec_list, dim=1), torch.cat(logits_list, dim=1)


class SequenceDiscriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim, time_steps, num_layers=3):
        super(SequenceDiscriminator, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        # Add Self Attention for advanced global pooling
        self.attn = TemporalSelfAttention(hidden_dim)
        # Use spectral normalization to enforce Lipschitz continuity for WGAN-GP
        self.fc = spectral_norm(nn.Linear(hidden_dim * time_steps, 1))

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.attn(out)
        out_flat = torch.flatten(out, start_dim=1)
        logits = self.fc(out_flat).squeeze(-1)  # Output an unbounded critic score for WGAN
        return logits, out


class BilateralLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(BilateralLSTMCell, self).__init__()
        self.hidden_dim = hidden_dim
        self.ln = nn.Linear(input_dim + hidden_dim + hidden_dim, 4 * hidden_dim, bias=False)

    def forward(self, x, h_self, c_self, h_coupled):
        combined = torch.cat([x, h_self, h_coupled], dim=1)
        gates = self.ln(combined)
        i_gate, f_gate, o_gate, c_tilde = gates.chunk(4, dim=1)

        i = torch.sigmoid(i_gate)
        f = torch.sigmoid(f_gate)
        o = torch.sigmoid(o_gate)
        c_ = torch.tanh(c_tilde)

        c_next = f * c_self + i * c_
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class BilateralGenerator(nn.Module):
    def __init__(self, noise_dim, hidden_dim, latent_dim, num_layers=3):
        super(BilateralGenerator, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        self.cl = nn.ModuleList([
            BilateralLSTMCell(
                input_dim=noise_dim if i == 0 else hidden_dim,
                hidden_dim=hidden_dim
            ) for i in range(num_layers)
        ])
        self.fc_out = nn.Linear(hidden_dim, latent_dim)

    def forward(self, noise_seq, h_coupled_states):
        batch_size, time_steps, _ = noise_seq.size()
        h_states = [torch.zeros(batch_size, self.hidden_dim, device=noise_seq.device) for _ in range(self.num_layers)]
        c_states = [torch.zeros(batch_size, self.hidden_dim, device=noise_seq.device) for _ in range(self.num_layers)]
        outputs = []

        for t in range(time_steps):
            x_t = noise_seq[:, t, :]
            for i in range(self.num_layers):
                h_cpl = h_coupled_states[i]
                h_states[i], c_states[i] = self.cl[i](x_t, h_states[i], c_states[i], h_cpl)
                x_t = h_states[i]

            out_t = torch.sigmoid(self.fc_out(x_t))
            outputs.append(out_t.unsqueeze(1))

        return torch.cat(outputs, dim=1), h_states


class JointGenerator(nn.Module):
    def __init__(self, c_noise_dim, d_noise_dim, hidden_dim, c_latent_dim, d_latent_dim, num_layers=3):
        super(JointGenerator, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        
        self.c_gen = BilateralGenerator(c_noise_dim, hidden_dim, c_latent_dim, num_layers)
        self.d_gen = BilateralGenerator(d_noise_dim, hidden_dim, d_latent_dim, num_layers)

        # Advanced Attention to resolve generator mode-collapse on patterns
        self.c_attn = TemporalSelfAttention(hidden_dim)
        self.d_attn = TemporalSelfAttention(hidden_dim)

    def forward(self, noise_c, noise_d):
        """
        Implements the step-by-step bilateral coupling required by EHR-M-GAN,
        enhanced with sequence-level Self-Attention for continuous distributions.
        """
        batch_size, time_steps, _ = noise_c.size()
        device = noise_c.device

        # Initialize hidden and cell states for both streams
        c_h = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        c_c = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        d_h = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        d_c = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]

        c_features_list = []
        d_features_list = []

        for t in range(time_steps):
            # 1. Capture current noise inputs
            noise_c_t = noise_c[:, t, :]
            noise_d_t = noise_d[:, t, :]

            # 2. Coupled inputs come from the OTHER stream's PREVIOUS hidden state
            c_h_coupled = d_h
            d_h_coupled = c_h

            # 3. Step C Generator Layers
            c_x = noise_c_t
            for i in range(self.num_layers):
                c_h[i], c_c[i] = self.c_gen.cl[i](c_x, c_h[i], c_c[i], c_h_coupled[i])
                c_x = c_h[i]
            
            c_features_list.append(c_x.unsqueeze(1))

            # 4. Step D Generator Layers
            d_x = noise_d_t
            for i in range(self.num_layers):
                d_h[i], d_c[i] = self.d_gen.cl[i](d_x, d_h[i], d_c[i], d_h_coupled[i])
                d_x = d_h[i]
            
            d_features_list.append(d_x.unsqueeze(1))

        # Apply Global Attention across the sequence to fix discrete pattern loss
        c_temporal = torch.cat(c_features_list, dim=1)
        d_temporal = torch.cat(d_features_list, dim=1)

        c_attn_out = self.c_attn(c_temporal)
        d_attn_out = self.d_attn(d_temporal)

        # Removed restrictive sigmoid constraint on generator output.
        # WGAN natively handles unconstrained generated latents (matches VAE true N(mu, sigma) mapping).
        fake_z_c = self.c_gen.fc_out(c_attn_out)
        fake_z_d = self.d_gen.fc_out(d_attn_out)

        return fake_z_c, fake_z_d


In [ ]:
%%writefile networks_v2.py
import torch
import torch.nn as nn
from torch.nn.utils import spectral_norm

class TemporalSelfAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalSelfAttention, self).__init__()
        self.query = nn.Linear(hidden_dim, hidden_dim // 8)
        self.key = nn.Linear(hidden_dim, hidden_dim // 8)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x shape: (batch_size, time_steps, hidden_dim)
        batch_size, time_steps, hidden_dim = x.size()
        
        proj_query = self.query(x) # B x T x C (where C = hidden_dim // 8)
        proj_key = self.key(x).permute(0, 2, 1) # B x C x T
        
        energy = torch.bmm(proj_query, proj_key) # B x T x T
        attention = torch.softmax(energy, dim=-1)
        
        proj_value = self.value(x) # B x T x H
        out = torch.bmm(attention, proj_value) # B x T x H
        
        out = self.gamma * out + x
        return out


class VAE_Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers=3):
        super(VAE_Encoder, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x_t, hidden_state):
        # x_t shape: [batch_size, 1, input_dim]
        out, new_hidden = self.lstm(x_t, hidden_state)
        out_squeeze = out.squeeze(1)

        mu = self.fc_mu(out_squeeze)
        logvar = self.fc_logvar(out_squeeze)

        # Defensive clamps to prevent numerical instability
        mu = torch.clamp(mu, min=-20.0, max=20.0)
        logvar = torch.clamp(logvar, min=-20.0, max=20.0)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z, mu, logvar, new_hidden


class VAE_Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim, num_layers=3):
        super(VAE_Decoder, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(latent_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z_t, hidden_state):
        # z_t shape: [batch_size, 1, latent_dim]
        out, new_hidden = self.lstm(z_t, hidden_state)
        logits = self.fc_out(out.squeeze(1))
        reconstruction = torch.sigmoid(logits)
        return reconstruction, logits, new_hidden


class AutoregressiveVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, enc_layers, dec_layers, time_steps):
        super(AutoregressiveVAE, self).__init__()
        self.time_steps = time_steps
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.encoder = VAE_Encoder(input_dim * 2, hidden_dim, latent_dim, enc_layers)
        self.decoder = VAE_Decoder(latent_dim, hidden_dim, input_dim, dec_layers)

    def forward(self, x):
        batch_size = x.size(0)
        device = x.device

        # Initialize hidden states
        enc_hidden = (torch.zeros(self.encoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.encoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))
        dec_hidden = (torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))

        c_prev = torch.zeros(batch_size, self.input_dim, device=device)

        rec_list, logits_list, mu_list, logvar_list, z_list = [], [], [], [], []

        for t in range(self.time_steps):
            x_t = x[:, t, :]
            c_sigmoid = torch.sigmoid(c_prev)
            x_hat = x_t - c_sigmoid

            enc_in = torch.cat([x_t, x_hat], dim=1).unsqueeze(1)
            z_t, mu_t, logvar_t, enc_hidden = self.encoder(enc_in, enc_hidden)

            z_in = z_t.unsqueeze(1)
            rec_t, logits_t, dec_hidden = self.decoder(z_in, dec_hidden)

            c_prev = logits_t

            rec_list.append(rec_t.unsqueeze(1))
            logits_list.append(logits_t.unsqueeze(1))
            mu_list.append(mu_t.unsqueeze(1))
            logvar_list.append(logvar_t.unsqueeze(1))
            z_list.append(z_t.unsqueeze(1))

        return torch.cat(rec_list, dim=1), torch.cat(logits_list, dim=1), \
            torch.cat(mu_list, dim=1), torch.cat(logvar_list, dim=1), torch.cat(z_list, dim=1)

    def reconstruct_decoder(self, z_seq):
        batch_size = z_seq.size(0)
        device = z_seq.device
        dec_hidden = (torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))

        rec_list, logits_list = [], []
        for t in range(self.time_steps):
            z_t = z_seq[:, t, :].unsqueeze(1)
            rec_t, logits_t, dec_hidden = self.decoder(z_t, dec_hidden)
            rec_list.append(rec_t.unsqueeze(1))
            logits_list.append(logits_t.unsqueeze(1))

        return torch.cat(rec_list, dim=1), torch.cat(logits_list, dim=1)


class SequenceDiscriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim, time_steps, num_layers=3):
        super(SequenceDiscriminator, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        # Tăng thêm 1 chiều đầu vào (input_dim + 1) để chứa kênh thống kê Minibatch StdDev
        self.lstm = nn.LSTM(input_dim + 1, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        # Add Self Attention for advanced global pooling
        self.attn = TemporalSelfAttention(hidden_dim)
        # Use spectral normalization to enforce Lipschitz continuity for WGAN-GP
        self.fc = spectral_norm(nn.Linear(hidden_dim * time_steps, 1))

    def forward(self, x):
        # x shape: [batch_size, time_steps, input_dim]
        batch_size, time_steps, channels = x.size()
        
        # Feature-wise Standard Deviation (Per-Sample Channel StdDev)
        # Computes variance across features/channels for each sample independently.
        # This completely avoids cross-sample gradient leakage during WGAN-GP double-backpropagation
        # and remains highly numerically stable since a single patient's features are never all constant.
        var = torch.var(x, dim=-1, keepdim=True, unbiased=False)
        std_mean = torch.sqrt(var + 1e-4)
        
        # Concatenate standard deviation to input features
        x_concat = torch.cat([x, std_mean], dim=-1) # shape: [batch_size, time_steps, input_dim + 1]

        out, _ = self.lstm(x_concat)
        out = self.attn(out)
        out_flat = torch.flatten(out, start_dim=1)
        logits = self.fc(out_flat).squeeze(-1)  # Output an unbounded critic score for WGAN
        return logits, out


class BilateralLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(BilateralLSTMCell, self).__init__()
        self.hidden_dim = hidden_dim
        self.ln = nn.Linear(input_dim + hidden_dim + hidden_dim, 4 * hidden_dim, bias=False)

    def forward(self, x, h_self, c_self, h_coupled):
        combined = torch.cat([x, h_self, h_coupled], dim=1)
        gates = self.ln(combined)
        i_gate, f_gate, o_gate, c_tilde = gates.chunk(4, dim=1)

        i = torch.sigmoid(i_gate)
        f = torch.sigmoid(f_gate)
        o = torch.sigmoid(o_gate)
        c_ = torch.tanh(c_tilde)

        c_next = f * c_self + i * c_
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class BilateralGenerator(nn.Module):
    def __init__(self, noise_dim, hidden_dim, latent_dim, num_layers=3):
        super(BilateralGenerator, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        self.cl = nn.ModuleList([
            BilateralLSTMCell(
                input_dim=noise_dim if i == 0 else hidden_dim,
                hidden_dim=hidden_dim
            ) for i in range(num_layers)
        ])
        self.fc_out = nn.Linear(hidden_dim, latent_dim)

    def forward(self, noise_seq, h_coupled_states):
        batch_size, time_steps, _ = noise_seq.size()
        h_states = [torch.zeros(batch_size, self.hidden_dim, device=noise_seq.device) for _ in range(self.num_layers)]
        c_states = [torch.zeros(batch_size, self.hidden_dim, device=noise_seq.device) for _ in range(self.num_layers)]
        outputs = []

        for t in range(time_steps):
            x_t = noise_seq[:, t, :]
            for i in range(self.num_layers):
                h_cpl = h_coupled_states[i]
                h_states[i], c_states[i] = self.cl[i](x_t, h_states[i], c_states[i], h_cpl)
                x_t = h_states[i]

            out_t = torch.sigmoid(self.fc_out(x_t))
            outputs.append(out_t.unsqueeze(1))

        return torch.cat(outputs, dim=1), h_states


class MappingNetwork(nn.Module):
    def __init__(self, input_dim, output_dim, num_layers=3):
        super(MappingNetwork, self).__init__()
        layers = []
        curr_dim = input_dim
        for _ in range(num_layers):
            layers.append(nn.Linear(curr_dim, output_dim))
            layers.append(nn.LeakyReLU(0.2, inplace=False))
            curr_dim = output_dim
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class JointGenerator(nn.Module):
    def __init__(self, c_noise_dim, d_noise_dim, hidden_dim, c_latent_dim, d_latent_dim, num_layers=3):
        super(JointGenerator, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        
        # Mapping Networks to transform standard Gaussian noise into intermediate latent space W (prevents mode collapse)
        self.c_map = MappingNetwork(c_noise_dim, c_noise_dim, num_layers=3)
        self.d_map = MappingNetwork(d_noise_dim, d_noise_dim, num_layers=3)

        self.c_gen = BilateralGenerator(c_noise_dim, hidden_dim, c_latent_dim, num_layers)
        self.d_gen = BilateralGenerator(d_noise_dim, hidden_dim, d_latent_dim, num_layers)

        self.c_attn = TemporalSelfAttention(hidden_dim)
        self.d_attn = TemporalSelfAttention(hidden_dim)

    def forward(self, noise_c, noise_d):
        """
        Implements the step-by-step bilateral coupling required by EHR-M-GAN,
        enhanced with sequence-level Self-Attention and mapping networks.
        """
        batch_size, time_steps, c_noise_dim = noise_c.size()
        d_noise_dim = noise_d.size(-1)
        device = noise_c.device

        # Pass noise through the mapping networks to disentangle the latent space
        noise_c_mapped = self.c_map(noise_c.view(-1, c_noise_dim)).view(batch_size, time_steps, c_noise_dim)
        noise_d_mapped = self.d_map(noise_d.view(-1, d_noise_dim)).view(batch_size, time_steps, d_noise_dim)

        # Initialize hidden and cell states for both streams
        c_h = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        c_c = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        d_h = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        d_c = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]

        c_features_list = []
        d_features_list = []

        for t in range(time_steps):
            # 1. Capture current noise inputs
            noise_c_t = noise_c_mapped[:, t, :]
            noise_d_t = noise_d_mapped[:, t, :]

            # 2. Coupled inputs come from the OTHER stream's PREVIOUS hidden state
            c_h_coupled = d_h
            d_h_coupled = c_h

            # 3. Step C Generator Layers
            c_x = noise_c_t
            for i in range(self.num_layers):
                c_h[i], c_c[i] = self.c_gen.cl[i](c_x, c_h[i], c_c[i], c_h_coupled[i])
                c_x = c_h[i]
            
            c_features_list.append(c_x.unsqueeze(1))

            # 4. Step D Generator Layers
            d_x = noise_d_t
            for i in range(self.num_layers):
                d_h[i], d_c[i] = self.d_gen.cl[i](d_x, d_h[i], d_c[i], d_h_coupled[i])
                d_x = d_h[i]
            
            d_features_list.append(d_x.unsqueeze(1))

        # Apply Global Attention across the sequence to fix discrete pattern loss
        c_temporal = torch.cat(c_features_list, dim=1)
        d_temporal = torch.cat(d_features_list, dim=1)

        c_attn_out = self.c_attn(c_temporal)
        d_attn_out = self.d_attn(d_temporal)

        # Co-scale latent bounds using tanh * 3.0 to match pretrained VAE distribution and prevent numerical explosion
        fake_z_c = torch.tanh(self.c_gen.fc_out(c_attn_out)) * 3.0
        fake_z_d = torch.tanh(self.d_gen.fc_out(d_attn_out)) * 3.0

        return fake_z_c, fake_z_d


In [ ]:
%%writefile ultils.py
import torch
import torch.nn.functional as F
import numpy as np


def nt_xent_loss(out_1, out_2, temperature=1.0):
    batch_size = out_1.shape[0]

    # ROOT FIX: Normalize the raw vectors BEFORE any dot products!
    # This prevents torch.exp() from blowing up to Infinity.
    out_1 = F.normalize(out_1, p=2, dim=-1)
    out_2 = F.normalize(out_2, p=2, dim=-1)

    out = torch.cat([out_1, out_2], dim=0)

    cov = torch.mm(out, out.t().contiguous())
    sim = torch.exp(cov / temperature)

    mask = ~torch.eye(2 * batch_size, device=out.device).bool()
    negatives = sim.masked_select(mask).view(2 * batch_size, -1)

    positives = torch.exp(torch.sum(out_1 * out_2, dim=-1) / temperature)
    positives = torch.cat([positives, positives], dim=0)

    # Add epsilons to prevent divide-by-zero and log(0)
    loss = -torch.log(positives / (negatives.sum(dim=-1) + 1e-8) + 1e-8)
    return loss.mean()


def kl_divergence(mu, logvar):
    # Sum over the latent dimension (-1) rather than the time sequence
    return -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1))


def feature_matching_loss(fake_features, real_features):
    mean_fake = torch.mean(fake_features, dim=0)
    mean_real = torch.mean(real_features, dim=0)
    std_fake = torch.sqrt(torch.var(fake_features, dim=0) + 1e-6)
    std_real = torch.sqrt(torch.var(real_features, dim=0) + 1e-6)

    loss_mean = torch.mean(torch.abs(mean_fake - mean_real))
    loss_std = torch.mean(torch.abs(std_fake - std_real))
    return loss_mean + loss_std


def renormlizer(data, max_val, min_val):
    # data = Value_norm * (Max - Min + 1e-8) + Min
    data = data * (max_val + 1e-8)
    data = data + min_val
    return data


def np_rounding(prob):
    y = np.round(prob)
    return y

In [ ]:
%%writefile metrics.py
import numpy as np
from sklearn.metrics.pairwise import rbf_kernel
import warnings
import torch


def discrete_probability_rmse(real_d, fake_d):
    """
    Calculates the Root Mean Square Error (RMSE) between the
    dimension-wise probabilities of the real and synthetic discrete data.
    """
    # Calculate probability of each code occurring across all samples and timesteps
    prob_real = np.mean(real_d, axis=(0, 1))
    prob_fake = np.mean(fake_d, axis=(0, 1))

    rmse = np.sqrt(np.mean((prob_real - prob_fake) ** 2))
    return rmse


def _mix_rbf_kernel(X, Y, sigmas, wts=None):
    if wts is None:
        wts = [1.0] * sigmas.shape[0]

    # Flatten dimensions similar to tf.tensordot(axes=[[1, 2], [1, 2]])
    X_flat = X.reshape(X.shape[0], -1)
    Y_flat = Y.reshape(Y.shape[0], -1)

    XX = torch.matmul(X_flat, X_flat.t())
    XY = torch.matmul(X_flat, Y_flat.t())
    YY = torch.matmul(Y_flat, Y_flat.t())

    X_sqnorms = torch.diag(XX)
    Y_sqnorms = torch.diag(YY)

    K_XX, K_XY, K_YY = 0., 0., 0.
    for sigma, wt in zip(sigmas, wts):
        gamma = 1 / (2 * sigma ** 2)
        K_XX += wt * torch.exp(-gamma * (-2 * XX + X_sqnorms.unsqueeze(1) + X_sqnorms.unsqueeze(0)))
        # X_sqnorms is (batch_x,), shape (m, 1) and Y_sqnorms is (batch_y,), shape (1, n)
        K_XY += wt * torch.exp(-gamma * (-2 * XY + X_sqnorms.unsqueeze(1) + Y_sqnorms.unsqueeze(0)))
        K_YY += wt * torch.exp(-gamma * (-2 * YY + Y_sqnorms.unsqueeze(1) + Y_sqnorms.unsqueeze(0)))

    return K_XX, K_XY, K_YY, sum(wts)


def _mmd2(K_XX, K_XY, K_YY, const_diagonal=False, biased=False):
    m = K_XX.shape[0]
    n = K_YY.shape[0]

    if biased:
        mmd2 = (torch.sum(K_XX) / (m * m)
                + torch.sum(K_YY) / (n * n)
                - 2 * torch.sum(K_XY) / (m * n))
    else:
        if const_diagonal is not False:
            trace_X = m * const_diagonal
            trace_Y = n * const_diagonal
        else:
            trace_X = torch.trace(K_XX)
            trace_Y = torch.trace(K_YY)

        mmd2 = ((torch.sum(K_XX) - trace_X) / (m * (m - 1))
                + (torch.sum(K_YY) - trace_Y) / (n * (n - 1))
                - 2 * torch.sum(K_XY) / (m * n))

    return mmd2


def mix_rbf_mmd2(X, Y, sigmas=None, wts=None, biased=True):
    if sigmas is None:
        sigmas = torch.tensor([1.0, 2.0, 4.0, 8.0, 16.0], device=X.device)
    K_XX, K_XY, K_YY, d = _mix_rbf_kernel(X, Y, sigmas, wts)
    return _mmd2(K_XX, K_XY, K_YY, const_diagonal=d, biased=biased)


def max_mean_discrepancy(real_data, syn_data, bandwidths=None):
    if not isinstance(real_data, torch.Tensor):
        X = torch.tensor(real_data, dtype=torch.float32)
    else:
        X = real_data.float()
        
    if not isinstance(syn_data, torch.Tensor):
        Y = torch.tensor(syn_data, dtype=torch.float32)
    else:
        Y = syn_data.float()

    if bandwidths is not None:
        if not isinstance(bandwidths, torch.Tensor):
            bandwidths = torch.tensor(bandwidths, dtype=torch.float32, device=X.device)

    # Move Y to X device in case they differ
    Y = Y.to(X.device)
    
    mmd2_value = mix_rbf_mmd2(X, Y, sigmas=bandwidths, biased=True) 

    # Prevent nan from sqrt of tiny negative numbers due to floating point precision
    mmd2_value = torch.clamp(mmd2_value, min=0.0) 
    
    return torch.sqrt(mmd2_value).item()


def pearson_correlation_error(real_data, fake_data):
    """
    Calculates the Mean Absolute Error between the Pearson correlation
    matrices of the real and synthetic features.
    """
    # Flatten temporal dimension to calculate overall feature-wise correlation
    # shape: (num_samples * time_steps, features)
    real_flat = real_data.reshape(-1, real_data.shape[2])
    fake_flat = fake_data.reshape(-1, fake_data.shape[2])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Calculate correlation matrices (features x features)
        corr_real = np.corrcoef(real_flat, rowvar=False)
        corr_fake = np.corrcoef(fake_flat, rowvar=False)

    # If a feature is entirely 0, correlation becomes NaN. Convert to 0.
    corr_real = np.nan_to_num(corr_real, nan=0.0)
    corr_fake = np.nan_to_num(corr_fake, nan=0.0)

    # Calculate mean absolute error between the matrices
    error = np.mean(np.abs(corr_real - corr_fake))
    return error


def evaluate_all(real_c, fake_c, real_d, fake_d):
    print("\n" + "=" * 40)
    print(" EHR-M-GAN EVALUATION METRICS")
    print("=" * 40)

    # 1. Continuous MMD
    mmd_score = max_mean_discrepancy(real_c, fake_c)
    print(f"Continuous MMD (Lower is better):      {mmd_score:.5f}")

    # 2. Discrete Probability RMSE
    rmse_score = discrete_probability_rmse(real_d, fake_d)
    print(f"Discrete Prob RMSE (Lower is better):  {rmse_score:.5f}")

    # 3. Continuous Feature Correlation Error
    corr_err_c = pearson_correlation_error(real_c, fake_c)
    print(f"Continuous Corr Error (Lower is better): {corr_err_c:.5f}")

    # 4. Discrete Feature Correlation Error
    corr_err_d = pearson_correlation_error(real_d, fake_d)
    print(f"Discrete Corr Error (Lower is better):   {corr_err_d:.5f}")
    print("=" * 40 + "\n")

    return {
        'mmd': float(mmd_score),
        'rmse': float(rmse_score),
        'corr_c': float(corr_err_c),
        'corr_d': float(corr_err_d)
    }

In [ ]:
# Dynamic testing module loader
import importlib.util
from ultils import np_rounding
from metrics import evaluate_all

def load_network_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def evaluate_checkpoint(model_version, checkpoint_path, real_c, real_d, num_samples=1000, batch_size=256):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load correct architecture
    if model_version == 'standard':
        net = load_network_module('net_std', 'networks_standard.py')
    elif model_version == 'v1':
        net = load_network_module('net_v1', 'networks_v1.py')
    elif model_version == 'v2':
        net = load_network_module('net_v2', 'networks_v2.py')
    else:
        raise ValueError("Invalid model version")
        
    c_dim, d_dim = real_c.shape[2], real_d.shape[2]
    time_steps = real_c.shape[1]
    latent_dim = 25
    noise_dim = min(int(c_dim / 2), int(d_dim / 2))
    
    c_vae = net.AutoregressiveVAE(c_dim, 512, latent_dim, 3, 3, time_steps).to(device)
    d_vae = net.AutoregressiveVAE(d_dim, 512, latent_dim, 3, 3, time_steps).to(device)
    joint_gen = net.JointGenerator(noise_dim, noise_dim, 512, latent_dim, latent_dim, 3).to(device)
    
    print(f"Loading {model_version.upper()} checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    c_vae.load_state_dict(checkpoint['c_vae'])
    d_vae.load_state_dict(checkpoint['d_vae'])
    
    # Support loading joint_gen directly (v2) or single generator states (standard/v1)
    if 'c_gen' in checkpoint:
        if hasattr(joint_gen, 'c_gen'):
            joint_gen.c_gen.load_state_dict(checkpoint['c_gen'])
            joint_gen.d_gen.load_state_dict(checkpoint['d_gen'])
    
    c_vae.eval(); d_vae.eval(); joint_gen.eval()
    
    c_gen_data, d_gen_data = [], []
    num_batches = int(np.ceil(num_samples / batch_size))
    
    with torch.no_grad():
        for _ in range(num_batches):
            noise_c = torch.randn(batch_size, time_steps, noise_dim, device=device)
            noise_d = torch.randn(batch_size, time_steps, noise_dim, device=device)
            
            fake_z_c, fake_z_d = joint_gen(noise_c, noise_d)
            fake_c_seq, _ = c_vae.reconstruct_decoder(fake_z_c)
            fake_d_seq, _ = d_vae.reconstruct_decoder(fake_z_d)
            
            c_gen_data.append(fake_c_seq.cpu().numpy())
            d_gen_data.append(fake_d_seq.cpu().numpy())
            
    c_gen_data = np.concatenate(c_gen_data, axis=0)[:num_samples]
    d_gen_data = np_rounding(np.concatenate(d_gen_data, axis=0))[:num_samples]
    
    indices = np.random.choice(real_c.shape[0], num_samples, replace=False)
    scores = evaluate_all(real_c[indices], c_gen_data, real_d[indices], d_gen_data)
    return scores

### 🏃‍♂️ Run 3-Model Side-by-Side Comparison:
Update the paths below to point to the checkpoints you downloaded/trained for **Standard (Baseline)**, **V1 (Self-Attention)**, and **V2 (Ours)** versions.

In [ ]:
# UPDATE THESE PATHS TO YOUR SAVED MODEL CHECKPOINTS
ckpt_standard = '/content/m3gan_standard_epoch_100.pth'
ckpt_v1 = '/content/m3gan_v1_epoch_100.pth'
ckpt_v2 = '/content/m3gan_v2_epoch_100.pth'

results = []

# 1. Evaluate Standard Model (Baseline)
if os.path.exists(ckpt_standard):
    scores_std = evaluate_checkpoint('standard', ckpt_standard, real_c_norm, real_d, num_samples=1000)
    scores_std.update({'model': 'Standard (Baseline)'})
    results.append(scores_std)
else:
    print("⚠️ Skip Standard (Baseline) - Checkpoint file not found.")
    
# 2. Evaluate V1 Model (Self-Attention)
if os.path.exists(ckpt_v1):
    scores_v1 = evaluate_checkpoint('v1', ckpt_v1, real_c_norm, real_d, num_samples=1000)
    scores_v1.update({'model': 'Neo M3GAN V1'})
    results.append(scores_v1)
else:
    print("⚠️ Skip Neo M3GAN V1 - Checkpoint file not found.")

# 3. Evaluate V2 Model (Ours: Mapping Network + Minibatch StdDev)
if os.path.exists(ckpt_v2):
    scores_v2 = evaluate_checkpoint('v2', ckpt_v2, real_c_norm, real_d, num_samples=1000)
    scores_v2.update({'model': 'Neo M3GAN V2 (Ours)'})
    results.append(scores_v2)
else:
    print("⚠️ Skip Neo M3GAN V2 (Ours) - Checkpoint file not found.")

### 📊 Visualizing Results & Statistical Report:
The cell below compiles a visual side-by-side comparison chart and displays a cleanly formatted markdown table comparing the models across all key metrics.

In [ ]:
if len(results) > 0:
    df = pd.DataFrame(results)
    # Reorder columns
    cols = ['model', 'mmd', 'rmse', 'corr_c', 'corr_d']
    df = df[cols]
    
    print("\n" + "="*60)
    print("              EHR-M-GAN 3-MODEL EFFICIENCY SUMMARY")
    print("="*60)
    # Avoid using tabulate library by hand-formatting a simple markdown table
    print(f"| {'Model':<25} | {'MMD':<10} | {'RMSE':<10} | {'Corr C':<10} | {'Corr D':<10} |")
    print(f"| {'-'*25} | {'-'*10} | {'-'*10} | {'-'*10} | {'-'*10} |")
    for _, row in df.iterrows():
        print(f"| {row['model']:<25} | {row['mmd']:<10.5f} | {row['rmse']:<10.5f} | {row['corr_c']:<10.5f} | {row['corr_d']:<10.5f} |")
    print("="*60 + "\n")
    
    # Plotting
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle('Side-by-Side Model Efficiency & Quality Comparison (Lower is Better)', fontsize=16, y=1.05)
    
    # 1. Continuous MMD
    axes[0].bar(df['model'], df['mmd'], color=['gray', 'orange', 'green'][:len(df)])
    axes[0].set_title('Continuous MMD')
    axes[0].set_ylabel('MMD Score')
    
    # 2. Discrete RMSE
    axes[1].bar(df['model'], df['rmse'], color=['gray', 'orange', 'green'][:len(df)])
    axes[1].set_title('Discrete Prob RMSE')
    axes[1].set_ylabel('RMSE Score')
    
    # 3. Continuous Correlation Error
    axes[2].bar(df['model'], df['corr_c'], color=['gray', 'orange', 'green'][:len(df)])
    axes[2].set_title('Continuous Correlation Error')
    axes[2].set_ylabel('Absolute Error')
    
    plt.tight_layout()
    plt.savefig('Output/analysis/3_model_efficiency_comparison.png', dpi=300)
    plt.show()
else:
    print("❌ No results to show! Please check your checkpoint paths.")